In [91]:
!wget https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv

--2026-10-10 11:41:40--  https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.111.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 278746 (272K) [text/plain]
Saving to: ‘course_lead_scoring_2026.csv.1’

course_lead_scoring 100%[===================>] 272.21K  --.-KB/s    in 0.09s   

2026-10-10 11:41:41 (2.87 MB/s) - ‘course_lead_scoring_2026.csv.1’ saved [278746/278746]



In [92]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [93]:
df = pd.read_csv('course_lead_scoring_2026.csv')

In [94]:
df.head(5)

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


In [95]:
df.dtypes

,0
lead_source,object
industry,object
employment_status,object
location,object
annual_income,float64
number_of_courses_viewed,int64
interaction_count,int64
lead_score,float64
converted,int64


In [96]:
numrical = ['annual_income', 'number_of_courses_viewed', 'interaction_count', 'lead_score', 'converted']
cat = ['lead_source', 'industry', 'employment_status', 'location']

In [97]:
df.isnull().sum()

,0
lead_source,148
industry,240
employment_status,194
location,208
annual_income,369
number_of_courses_viewed,0
interaction_count,0
lead_score,35
converted,0


In [98]:
df[numrical] = df[numrical].fillna(0.0)
df[cat] = df[cat].fillna('NA')
df.isnull().sum()

,0
lead_source,0
industry,0
employment_status,0
location,0
annual_income,0
number_of_courses_viewed,0
interaction_count,0
lead_score,0
converted,0


In [99]:
df['industry'].value_counts()

,count
industry,
technology,1173
retail,925
healthcare,840
finance,720
education,639
manufacturing,463
NA,240


In [100]:
df[numrical].corr()

,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
annual_income,1.000000,0.161300,0.122842,0.229496,0.189652
number_of_courses_viewed,0.161300,1.000000,0.721609,0.757204,0.395038
interaction_count,0.122842,0.721609,1.000000,0.915746,0.448624
lead_score,0.229496,0.757204,0.915746,1.000000,0.483246
converted,0.189652,0.395038,0.448624,0.483246,1.000000


In [101]:
from sklearn.model_selection import train_test_split

In [102]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=42

)
y_train = df_train.converted.values
y_val = df_val.converted.values
y_test = df_test.converted.values

del df_train['converted']
del df_val['converted']
del df_test['converted']


In [103]:
from sklearn.metrics import mutual_info_score

In [104]:
def mutual_inf(s):
  return mutual_info_score(s, df_full_train.converted)

score = df_full_train[cat].apply(mutual_inf)
round(score, 2)


,0
lead_source,0.03
industry,0.00
employment_status,0.02
location,0.00


In [105]:
from sklearn.feature_extraction import DictVectorizer

In [106]:
dv = DictVectorizer(sparse=False)
x_train = dv.fit_transform(df_train.to_dict(orient='records'))
x_val = dv.transform(df_val.to_dict(orient='records'))

In [107]:
from sklearn.linear_model import LogisticRegression

In [108]:
model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)

In [109]:
model.fit(x_train, y_train)
y_pred = model.predict(x_val)
score = (y_pred == y_val).mean()
score

np.float64(0.645)

In [117]:
features1 = df_train.columns.tolist()
features1.remove('lead_source')

dv = DictVectorizer(sparse=False)
x_train = dv.fit_transform(df_train[features1].to_dict(orient='records'))
x_val = dv.transform(df_val[features1].to_dict(orient='records'))

model.fit(x_train, y_train)
y_pred = model.predict(x_val)
score1 = (y_pred == y_val).mean()
score1 - score

np.float64(-0.0030000000000000027)

In [118]:
features2 = df_train.columns.tolist()
features2.remove('number_of_courses_viewed')

dv = DictVectorizer(sparse=False)
x_train = dv.fit_transform(df_train[features2].to_dict(orient='records'))
x_val = dv.transform(df_val[features2].to_dict(orient='records'))

model.fit(x_train, y_train)
y_pred = model.predict(x_val)
score2 = (y_pred == y_val).mean()
score2 , score2 - score

(np.float64(0.643), np.float64(-0.0020000000000000018))

In [119]:
features3 = df_train.columns.tolist()
features3.remove('interaction_count')

dv = DictVectorizer(sparse=False)
x_train = dv.fit_transform(df_train[features3].to_dict(orient='records'))
x_val = dv.transform(df_val[features3].to_dict(orient='records'))

model.fit(x_train, y_train)
y_pred = model.predict(x_val)
score3 = (y_pred == y_val).mean()
score3 , score3 - score

(np.float64(0.601), np.float64(-0.04400000000000004))

In [126]:
dv = DictVectorizer(sparse=False)
x_train = dv.fit_transform(df_train.to_dict(orient='records'))
x_val = dv.transform(df_val.to_dict(orient='records'))

scores = {}
for c in [0.000001, 0.00001, 0.0001, 0.001]:
  model = LogisticRegression(solver='liblinear', C=c, max_iter=1000, random_state=42)
  model.fit(x_train, y_train)
  y_pred = model.predict(x_val)
  score = (y_pred == y_val).mean()
  scores[c] = round(score, 3)
scores

{1e-06: np.float64(0.598),
 1e-05: np.float64(0.598),
 0.0001: np.float64(0.613),
 0.001: np.float64(0.645)}